In [2]:
import os
import re
import json
import time
import warnings
from pathlib import Path

# Tell PyTorch to report CUDA errors synchronously
# Without this, CUDA errors appear at the wrong line
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

warnings.filterwarnings("ignore", category=UserWarning)

import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration, pipeline
from jiwer import wer, cer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

plt.style.use("dark_background")
pd.set_option("display.max_colwidth", 120)

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("C:/xcas-ga-comms-assistant")
AUDIO_DIR    = PROJECT_ROOT / "tartan_data/kbtp/2020/10/10-22-20_audio"
CALLOUTS_CSV = PROJECT_ROOT / "data/interim/adsb_with_callouts_2020-10-22.csv"
MANIFEST_CSV = PROJECT_ROOT / "data/interim/manifest_2020-10-22.csv"
OUTPUT_DIR   = PROJECT_ROOT / "data/interim"
RESULTS_DIR  = PROJECT_ROOT / "outputs"
RESULTS_DIR.mkdir(exist_ok=True)

# ── Blackwell-safe dtype selection ────────────────────────────────────────────
# SM 12.0 (Blackwell) has incomplete float16 kernel support in PyTorch 2.11
# bfloat16 sometimes also fails — float32 is safe but uses more VRAM
# We test in order: bfloat16 → float32, pick first that works

def get_safe_dtype_and_device():
    """
    Detect GPU capability and return the safest dtype for inference.
    Blackwell (SM 12.x): prefer bfloat16 with eager attention.
    If GPU fails: fall back to CPU float32.
    """
    if not torch.cuda.is_available():
        print("No GPU — using CPU float32")
        return "cpu", torch.float32

    major, minor = torch.cuda.get_device_capability(0)
    sm = major * 10 + minor
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

    print(f"GPU    : {gpu_name}")
    print(f"SM     : {major}.{minor}  (SM{sm})")
    print(f"VRAM   : {vram_gb:.1f} GB")
    print(f"CUDA   : {torch.version.cuda}")

    if major >= 12:
        print("Blackwell detected → using bfloat16 + eager attention")
        return "cuda", torch.bfloat16
    elif major >= 8:
        print("Ampere/Ada detected → using float16")
        return "cuda", torch.float16
    else:
        print("Older GPU → using float32")
        return "cuda", torch.float32

DEVICE, DTYPE = get_safe_dtype_and_device()
print(f"\nDevice : {DEVICE}")
print(f"Dtype  : {DTYPE}")

# ── File inventory ─────────────────────────────────────────────────────────────
wav_files = sorted(AUDIO_DIR.glob("*.wav"))
txt_files = sorted(AUDIO_DIR.glob("*.txt"))
print(f"\nWAV files : {len(wav_files)}")
print(f"TXT files : {len(txt_files)}")

GPU    : NVIDIA GeForce RTX 5070 Ti Laptop GPU
SM     : 12.0  (SM120)
VRAM   : 12.8 GB
CUDA   : 12.8
Blackwell detected → using bfloat16 + eager attention

Device : cuda
Dtype  : torch.bfloat16

WAV files : 52
TXT files : 52


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL LOADING — with three Blackwell-specific fixes:
#
# Fix 1: attn_implementation="eager"
#   Disables FlashAttention-2 which has incomplete SM_120 support.
#   "eager" = standard PyTorch attention (slower, but correct on all hardware)
#   FlashAttention uses custom CUDA kernels — if those kernels aren't compiled
#   for SM_120, you get cudaErrorIllegalAddress at the attention computation.
#
# Fix 2: low_cpu_mem_usage=True
#   Loads weights shard-by-shard rather than all at once.
#   Prevents OOM during loading on systems near VRAM limit.
#
# Fix 3: torch.backends.cuda.enable_flash_sdp(False)
#   Disables scaled dot product attention flash implementation globally.
#   Belt-and-suspenders with Fix 1.
# ═══════════════════════════════════════════════════════════════════════════════

# Disable flash SDP globally before any model operations
if DEVICE == "cuda":
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    print("✅ FlashAttention disabled (Blackwell compatibility)")

MODEL_ID = "jlvdoorn/whisper-small.en-atco2-asr"
print(f"\nLoading: {MODEL_ID}")
print("First run downloads ~500MB — cached after that\n")

t0 = time.time()

processor = WhisperProcessor.from_pretrained(MODEL_ID)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype        = DTYPE,
    low_cpu_mem_usage  = True,
    attn_implementation= "eager",    # Fix 1: no FlashAttention
)
model = model.to(DEVICE)
model.eval()

# Disable KV cache (sometimes causes memory issues on new architectures)
model.config.use_cache = True   # keep enabled for speed, disable if crash

load_time = time.time() - t0

# Print model architecture so you understand what you loaded
print(f"✅ Loaded in {load_time:.1f}s")
print(f"\nArchitecture summary:")
print(f"  Type              : {model.config.model_type}")
print(f"  Encoder layers    : {model.config.encoder_layers}")
print(f"  Decoder layers    : {model.config.decoder_layers}")
print(f"  Attention heads   : {model.config.encoder_attention_heads}")
print(f"  Hidden dim (d_model): {model.config.d_model}")
print(f"  Mel filterbanks   : {model.config.num_mel_bins}")
print(f"  Max audio window  : {model.config.max_source_positions//100}s")
print(f"  Vocabulary size   : {model.config.vocab_size:,} tokens")
print(f"  Total parameters  : {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

if DEVICE == "cuda":
    used = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"\nVRAM used  : {used:.2f} GB")
    print(f"VRAM free  : {total-used:.2f} GB")

✅ FlashAttention disabled (Blackwell compatibility)

Loading: jlvdoorn/whisper-small.en-atco2-asr
First run downloads ~500MB — cached after that



Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1698.28it/s]


✅ Loaded in 3.1s

Architecture summary:
  Type              : whisper
  Encoder layers    : 12
  Decoder layers    : 12
  Attention heads   : 12
  Hidden dim (d_model): 768
  Mel filterbanks   : 80
  Max audio window  : 15s
  Vocabulary size   : 51,864 tokens
  Total parameters  : 241.7M

VRAM used  : 0.50 GB
VRAM free  : 12.32 GB


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# GPU SMOKE TEST
# Run a 3-second synthetic audio through the model BEFORE loading real files.
# If this crashes → GPU issue. If it passes → real audio will work.
# This isolates hardware problems from data problems.
# ═══════════════════════════════════════════════════════════════════════════════

print("Running GPU smoke test with synthetic audio...")

# Create 3 seconds of white noise at 16kHz
# White noise exercises the same code paths as real audio
test_audio = np.random.randn(16000 * 3).astype(np.float32) * 0.01

try:
    # Process through feature extractor manually (not pipeline)
    # This lets us test each step independently
    
    # Step 1: Audio → Mel spectrogram features
    inputs = processor(
        test_audio,
        sampling_rate=16000,
        return_tensors="pt"
    )
    input_features = inputs.input_features.to(DEVICE, dtype=DTYPE)
    print(f"  Step 1 ✅ Mel features: {input_features.shape}")
    # Expected: torch.Size([1, 80, 3000]) = batch × mel_bins × time_frames
    # 80 mel bins is Whisper's filterbank configuration
    # 3000 time frames = 30 seconds × 100 frames/second

    # Step 2: Encoder forward pass
    with torch.no_grad():
        encoder_out = model.model.encoder(input_features)
    print(f"  Step 2 ✅ Encoder output: {encoder_out.last_hidden_state.shape}")
    # Expected: torch.Size([1, 1500, 512]) = batch × seq × hidden_dim

    # Step 3: Full generation (encoder + decoder)
    with torch.no_grad():
        generated = model.generate(
            input_features,
            max_new_tokens=10,
        )
    decoded = processor.batch_decode(generated, skip_special_tokens=True)
    print(f"  Step 3 ✅ Generation works. Output (noise→): '{decoded[0]}'")

    print(f"\n✅ GPU smoke test PASSED — safe to run on real audio")
    GPU_WORKS = True

except Exception as e:
    print(f"\n✗ GPU smoke test FAILED: {e}")
    print("→ Falling back to CPU for inference")
    DEVICE = "cpu"
    DTYPE  = torch.float32
    model  = model.to("cpu").float()
    GPU_WORKS = False
    print("→ CPU fallback active (slower but correct)")

Running GPU smoke test with synthetic audio...
  Step 1 ✅ Mel features: torch.Size([1, 80, 3000])


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers

  Step 2 ✅ Encoder output: torch.Size([1, 1500, 768])
  Step 3 ✅ Generation works. Output (noise→): ' you'

✅ GPU smoke test PASSED — safe to run on real audio


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# VAD FOR RADIO RECORDINGS
#
# Problem recap: all 52 files passed -20 dBFS threshold because the radio
# receiver adds a constant broadband squelch noise floor across all recordings.
# File 12.wav (rms=0.0002) is near-silence but still passes.
#
# Solution: measure energy RELATIVE to each file's own noise floor.
# Speech = short high-energy bursts, typically 10-20 dB above the noise floor.
# Radio noise = constant low-level broadband signal.
#
# Also: you told us 3.wav has ~4 seconds of real speech ("take-off runway 26,
# ...butler county"). This means even very sparse files are valuable.
# So we set a LOW bar — we want to keep any file with detectable speech.
# ═══════════════════════════════════════════════════════════════════════════════

def compute_vad_profile(wav_path: Path) -> dict:
    """
    Compute voice activity profile for a radio recording.
    
    Key insight: radio recordings have a noise floor that is NOT silence.
    We must measure SNR relative to EACH FILE'S OWN baseline,
    not against an absolute dBFS threshold.
    
    Returns statistics needed to decide whether to transcribe
    and to guide the transcription (speech timestamps).
    """
    # Load at 16kHz (Whisper's required rate)
    audio, sr = librosa.load(str(wav_path), sr=16000, mono=True)
    duration_s = len(audio) / sr

    # ── Frame-level RMS energy ─────────────────────────────────────────────────
    frame_length = 1024   # ~64ms at 16kHz — short enough to catch 4s speech bursts
    hop_length   = 256    # ~16ms hop — good time resolution

    rms = librosa.feature.rms(
        y=audio, frame_length=frame_length, hop_length=hop_length
    )[0]
    rms_db = 20 * np.log10(rms + 1e-10)

    # ── Noise floor: median of bottom 30% of frames ───────────────────────────
    # 30th percentile is robustly inside the noise region
    # even for dense recordings with lots of speech
    noise_floor_db  = float(np.percentile(rms_db, 30))
    noise_floor_rms = 10 ** (noise_floor_db / 20)

    # ── SNR per frame relative to this file's noise floor ────────────────────
    snr_db = rms_db - noise_floor_db

    # ── Speech detection thresholds ───────────────────────────────────────────
    # 8 dB above noise floor = clearly above background
    # Radio speech typically 12-20 dB above squelch noise
    SPEECH_SNR_DB     = 8.0    # dB above floor to count as speech frame
    MIN_SPEECH_FRAMES = 5      # at least 5 frames (5 × 16ms = ~80ms) = real speech

    speech_mask   = snr_db > SPEECH_SNR_DB
    n_speech_frames = int(np.sum(speech_mask))
    speech_fraction = float(np.mean(speech_mask))

    # ── Convert speech frame indices to timestamps ────────────────────────────
    # These tell us WHERE in the file speech occurs
    frame_times = librosa.frames_to_time(
        np.where(speech_mask)[0],
        sr=sr, hop_length=hop_length
    )

    # Find contiguous speech segments
    speech_segments = []
    if len(frame_times) > 0:
        gaps = np.diff(frame_times)
        segment_breaks = np.where(gaps > 0.5)[0] + 1  # 500ms gap = new segment
        segment_frames = np.split(frame_times, segment_breaks)
        for seg in segment_frames:
            if len(seg) > 0:
                speech_segments.append({
                    "start": float(seg[0]),
                    "end"  : float(seg[-1]) + 0.064,  # add one frame duration
                    "duration": float(seg[-1] - seg[0]) + 0.064
                })

    has_speech = (n_speech_frames >= MIN_SPEECH_FRAMES and
                  float(np.max(snr_db)) > SPEECH_SNR_DB)

    return {
        "wav_file"       : wav_path.name,
        "duration_s"     : duration_s,
        "noise_floor_db" : noise_floor_db,
        "peak_snr_db"    : float(np.max(snr_db)),
        "mean_snr_db"    : float(np.mean(snr_db[speech_mask]) if speech_mask.any() else 0),
        "n_speech_frames": n_speech_frames,
        "speech_fraction": speech_fraction,
        "n_speech_segs"  : len(speech_segments),
        "speech_segments": speech_segments,
        "has_speech"     : has_speech,
        "total_speech_s" : sum(s["duration"] for s in speech_segments),
    }


# ── Profile all files ─────────────────────────────────────────────────────────
print("VAD Profile — all 52 WAV files")
print(f"{'File':8s}  {'noise_fl':>8s}  {'peak_snr':>8s}  "
      f"{'speech%':>8s}  {'n_segs':>6s}  {'speech_s':>8s}  status")
print("-"*72)

vad_results = []
for wav_path in sorted(wav_files):
    v = compute_vad_profile(wav_path)
    vad_results.append(v)
    status = "✅" if v["has_speech"] else "✗ skip"
    print(f"  {v['wav_file']:8s}  {v['noise_floor_db']:8.1f}  "
          f"{v['peak_snr_db']:8.1f}  "
          f"{v['speech_fraction']:8.1%}  "
          f"{v['n_speech_segs']:6d}  "
          f"{v['total_speech_s']:8.1f}s  {status}")

df_vad = pd.DataFrame(vad_results)
speech_files = df_vad[df_vad["has_speech"]]["wav_file"].tolist()

print(f"\n{'='*72}")
print(f"Files to transcribe : {df_vad['has_speech'].sum()}")
print(f"Files to skip       : {(~df_vad['has_speech']).sum()}")
print(f"Total speech audio  : {df_vad['total_speech_s'].sum():.1f}s "
      f"({df_vad['total_speech_s'].sum()/60:.1f} min)")

# Check 3.wav specifically — you heard 4s of speech
wav3 = df_vad[df_vad["wav_file"]=="3.wav"].iloc[0]
print(f"\n3.wav check (you heard 4s of real speech):")
print(f"  Peak SNR    : {wav3['peak_snr_db']:.1f} dB")
print(f"  Speech segs : {wav3['n_speech_segs']}")
print(f"  Total speech: {wav3['total_speech_s']:.1f}s")
print(f"  Detected    : {'✅ YES' if wav3['has_speech'] else '✗ NO — threshold too strict'}")
if wav3["speech_segments"]:
    print(f"  Segments    : {wav3['speech_segments'][:3]}")

VAD Profile — all 52 WAV files
File      noise_fl  peak_snr   speech%  n_segs  speech_s  status
------------------------------------------------------------------------
  10.wav       -78.8      73.2     10.8%      95      30.9s  ✅
  12.wav       -78.7      73.4     13.9%     510     200.4s  ✅
  13.wav       -77.0      71.6     21.8%     252     256.1s  ✅
  14.wav       -73.5      68.1     28.0%      71     263.5s  ✅
  15.wav       -75.7      70.3     21.4%      80     201.3s  ✅
  16.wav       -74.3      69.0     28.8%      67     271.3s  ✅
  17.wav       -77.1      71.6     12.0%      78      51.9s  ✅
  18.wav       -74.5      69.2     12.6%      77     135.0s  ✅
  19.wav       -73.1      67.8     20.4%      87     205.0s  ✅
  20.wav       -76.9      71.6      9.5%     172      71.6s  ✅
  21.wav       -78.5      73.1     11.1%     413     187.0s  ✅
  23.wav       -75.0      69.6     29.3%      65     272.5s  ✅
  24.wav       -78.3      72.8     10.0%     144      59.6s  ✅
  25.wav    

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# BUILD PIPELINE — with hallucination suppression
#
# 📚 Two sources of hallucination we suppress:
#
# 1. Silence hallucination → condition_on_prev_tokens=False
#    By default Whisper conditions each 30s chunk on the previous chunk's text.
#    On silence, the previous chunk might be hallucinated YouTube text.
#    That text then "primes" the next chunk to continue hallucinating.
#    Setting False makes each chunk independent.
#
# 2. Repetition loops → no_repeat_ngram_size=3
#    "Delta Delta Delta..." loops happen when the decoder gets stuck.
#    Forbidding any 3-gram from repeating breaks the loop.
#    3-gram is chosen to not interfere with legitimate repeated words
#    like "Butler Traffic ... Butler Traffic" (the callout structure).
#
# 3. Temperature fallback → temperature=(0.0, 0.2, 0.4)
#    Whisper's own anti-hallucination: if decoding confidence is low,
#    it retries with higher temperature (more randomness).
#    This breaks repetition loops that survive no_repeat_ngram_size.
# ═══════════════════════════════════════════════════════════════════════════════

# Build pipeline
asr_pipeline = pipeline(
    task                = "automatic-speech-recognition",
    model               = model,
    tokenizer           = processor.tokenizer,
    feature_extractor   = processor.feature_extractor,
    chunk_length_s      = 30,
    stride_length_s     = 5,
    device              = 0 if DEVICE=="cuda" else -1,
    torch_dtype         = DTYPE,
    return_timestamps   = True,
    generate_kwargs     = {
        "condition_on_prev_tokens": False,    # no cross-chunk hallucination
        "no_repeat_ngram_size"    : 3,        # break repetition loops
        "temperature"             : (0.0, 0.2, 0.4),  # fallback on low confidence
        "compression_ratio_threshold": 1.35,  # flag suspicious repetition
        "logprob_threshold"       : -1.0,     # flag low-confidence outputs
    }
)

print("✅ Pipeline built")
print("   Hallucination suppression:")
print("   - condition_on_prev_tokens=False  (no cross-chunk priming)")
print("   - no_repeat_ngram_size=3          (break repetition loops)")
print("   - temperature fallback (0, 0.2, 0.4) (retry on low confidence)")

# ── Test on single file BEFORE batch run ─────────────────────────────────────
# Use 3.wav — smallest file (you heard 4s of real speech in it)
# If this works without CUDA crash, all files will work
test_wav = AUDIO_DIR / "3.wav"

print(f"\nSingle-file test: {test_wav.name}")
print("(You heard 'take-off runway 26... butler county' in this file)")
print("-"*55)

audio_test, _ = librosa.load(str(test_wav), sr=16000, mono=True)
print(f"Audio loaded: {len(audio_test)/16000:.1f}s, dtype={audio_test.dtype}")

t0 = time.time()
try:
    result = asr_pipeline(audio_test)
    elapsed = time.time() - t0
    rtf = elapsed / (len(audio_test)/16000)

    print(f"✅ Transcription succeeded in {elapsed:.1f}s  (RTF={rtf:.2f})")
    print(f"\nTranscript:\n{result['text'][:500]}")

    if result.get("chunks"):
        print(f"\nTimestamped chunks (first 5):")
        for chunk in result["chunks"][:5]:
            ts = chunk.get("timestamp", (0, 0))
            txt = chunk["text"].strip()
            if txt:
                print(f"  [{ts[0]:6.1f}s – {ts[1]:6.1f}s]  {txt}")

    # Does it contain your expected phrase?
    keywords = ["runway", "26", "butler", "takeoff", "take-off", "traffic"]
    found = [kw for kw in keywords if kw.lower() in result["text"].lower()]
    print(f"\nExpected keywords found: {found}")

except Exception as e:
    elapsed = time.time() - t0
    print(f"✗ FAILED after {elapsed:.1f}s")
    print(f"Error: {type(e).__name__}: {str(e)[:200]}")
    print("\n→ If CUDA crash: try setting DEVICE='cpu' in Cell 1 and rerun")

`torch_dtype` is deprecated! Use `dtype` instead!
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Passing `generation_config` together with generation-related arguments=({'no_repeat_ngram_size'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Pipeline built
   Hallucination suppression:
   - condition_on_prev_tokens=False  (no cross-chunk priming)
   - no_repeat_ngram_size=3          (break repetition loops)
   - temperature fallback (0, 0.2, 0.4) (retry on low confidence)

Single-file test: 3.wav
(You heard 'take-off runway 26... butler county' in this file)
-------------------------------------------------------
Audio loaded: 246.1s, dtype=float32
✅ Transcription succeeded in 11.6s  (RTF=0.05)

Transcript:
 takeoff runway two six cleared for departure runway five Okay. Okay. Okay. Okay. So we're going to start with the next one. So we'll start with the next one, which is the next one is the next one. So we have a next one which is the next one. So we have a next one which is the next one which is the next one which is the next one which is the next one which is . . . . . Okay. ... ... ... ...

Timestamped chunks (first 5):
  [   0.0s –  165.0s]  takeoff runway two six cleared for departure runway five Okay. Okay. Okay. 

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SEGMENT-LEVEL TRANSCRIPTION
#
# Architecture change: instead of feeding whole files to Whisper,
# we use VAD timestamps to extract only speech segments.
# Each segment is transcribed independently as a short clip.
#
# 📚 Why this works:
#   - A 5s speech burst → 5s of audio → model must produce ~5s of tokens
#   - No "padding space" to hallucinate into
#   - Much faster (process only 12% of audio, not 100%)
#   - Matches how Whisper was designed: short utterance → text
#
# 📚 Whisper on short clips (<30s):
#   For clips under 30s, Whisper pads with silence to fill the 30s window.
#   The padding is masked so the model knows it's padding — no hallucination.
#   This is safe and correct for segments up to 30s.
#   For segments >30s: use chunk_length_s=30 with stride.
#
# Pipeline for each WAV file:
#   1. Load full audio at 16kHz
#   2. For each VAD speech segment:
#      a. Extract audio slice [start:end] + small buffer
#      b. Run Whisper on that slice (no chunking needed for <30s clips)
#      c. Filter out hallucinations (short output, keyword check)
#   3. Assemble all segment transcripts with timestamps
# ═══════════════════════════════════════════════════════════════════════════════

# Known Whisper hallucination phrases on silence/noise
# These appear consistently across files and are never real ATC speech
HALLUCINATION_PATTERNS = [
    r"thank you (so much )?for (joining|watching|listening)",
    r"see you (in the )?(next|later)",
    r"bye[\-\s]?bye",
    r"(okay[,\.]?\s*){3,}",
    r"(we (are|will|have) (now|start|ready)[^\n]*){2,}",
    r"(the next one[^\n]*){2,}",
    r"(runway (two six|zero eight)[^\n]*){3,}",
    r"(\.\s*){4,}",
    r"(delta\s*){3,}",

    # ── NEW: single-word noise decodes ────────────────────────────────────
    # Whisper consistently outputs these on short noise bursts
    r"^you\.?$",           # exactly "you" or "you."
    r"^(you\s*){1,3}$",   # "you you you"
    r"^\s*(the|a|an)\s*$", # articles only
    r"^(uh+|um+|ah+|mm+)\.?$",  # filler sounds
]
HALLUCINATION_RE = re.compile(
    "|".join(HALLUCINATION_PATTERNS),
    re.IGNORECASE | re.DOTALL
)

# Keywords that CONFIRM real ATC speech
ATC_KEYWORDS = [
    "runway", "traffic", "butler", "inbound", "outbound", "departing",
    "final", "downwind", "base", "crosswind", "cleared", "takeoff",
    "take-off", "landing", "pattern", "miles", "north", "south",
    "east", "west", "cessna", "piper", "november", "alpha", "bravo",
    "charlie", "delta", "echo", "foxtrot", "golf", "hotel", "india",
    "juliet", "kilo", "lima", "mike", "oscar", "papa", "quebec",
    "romeo", "sierra", "tango", "uniform", "victor", "whiskey",
    "xray", "yankee", "zulu",
]

def is_hallucination(text: str) -> bool:
    """
    Detect Whisper hallucination using two signals:
    1. Matches known hallucination patterns (YouTube endings, loops)
    2. Contains NO aviation keywords despite being 'transcription'
    """
    if not text or len(text.strip()) < 3:
        return True
    text_clean = text.strip()
    # Check for hallucination patterns
    if HALLUCINATION_RE.search(text_clean):
        return True
    # Very short output (< 3 words) with no ATC keywords = likely noise decode
    words = text_clean.lower().split()
    if len(words) <= 2:
        has_atc = any(kw in text_clean.lower() for kw in ATC_KEYWORDS)
        if not has_atc:
            return True
    return False


def transcribe_segments(wav_path: Path,
                         vad_row: dict,
                         min_seg_duration: float = 1.5,
                         buffer_s: float = 0.3,
                         max_seg_duration: float = 30.0) -> dict:
    """
    Transcribe only the VAD-detected speech segments of a WAV file.
    
    Parameters
    ──────────
    min_seg_duration : skip segments shorter than this (noise bursts)
    buffer_s         : add this many seconds before/after each segment
                       to capture clipped word starts/ends
    max_seg_duration : segments longer than this get chunked (rare for CTAF)
    
    Returns dict with:
        segments     : list of {start, end, text, duration, is_hallucination}
        full_text    : clean concatenated transcript
        n_segments   : total VAD segments attempted
        n_transcribed: segments with non-hallucinated output
        rtf          : real-time factor for speech-only audio
    """
    # Load full file at 16kHz
    audio_full, sr = librosa.load(str(wav_path), sr=16000, mono=True)
    total_duration  = len(audio_full) / sr

    speech_segs = vad_row.get("speech_segments", [])

    # Filter to meaningful segments
    segs_to_process = [
        s for s in speech_segs
        if s["duration"] >= min_seg_duration
    ]

    results = []
    t_start_total = time.time()
    total_speech_audio = 0.0

    for seg in segs_to_process:
        # Add buffer around segment (catch clipped beginnings)
        seg_start = max(0.0, seg["start"] - buffer_s)
        seg_end   = min(total_duration, seg["end"] + buffer_s)
        seg_dur   = seg_end - seg_start

        # Extract audio slice
        start_sample = int(seg_start * sr)
        end_sample   = int(seg_end * sr)
        audio_slice  = audio_full[start_sample:end_sample]

        total_speech_audio += seg_dur

        try:
            if seg_dur <= max_seg_duration:
                # Short segment: direct inference (no chunking needed)
                # Processor converts to mel spectrogram
                inputs = processor(
                    audio_slice,
                    sampling_rate=16000,
                    return_tensors="pt"
                )
                input_features = inputs.input_features.to(DEVICE, dtype=DTYPE)

                with torch.no_grad():
                    generated_ids = model.generate(
                        input_features,
                        # Suppress timestamp tokens for short clips
                        # (we already have VAD timestamps — no need for Whisper timestamps)
                        return_timestamps=False,
                        no_repeat_ngram_size=3,
                        condition_on_prev_tokens=False,
                    )

                text = processor.batch_decode(
                    generated_ids, skip_special_tokens=True
                )[0].strip()

            else:
                # Long segment (rare): use pipeline with chunking
                result = asr_pipeline(audio_slice)
                text = result["text"].strip()

            hallucinated = is_hallucination(text)
            has_atc_kw   = any(kw in text.lower() for kw in ATC_KEYWORDS)

            results.append({
                "start"           : seg_start,
                "end"             : seg_end,
                "duration"        : seg_dur,
                "text"            : text,
                "is_hallucination": hallucinated,
                "has_atc_keyword" : has_atc_kw,
            })

        except Exception as e:
            results.append({
                "start"           : seg_start,
                "end"             : seg_end,
                "duration"        : seg_dur,
                "text"            : "",
                "is_hallucination": True,
                "has_atc_keyword" : False,
                "error"           : str(e)[:100],
            })

    elapsed = time.time() - t_start_total
    rtf = elapsed / total_speech_audio if total_speech_audio > 0 else 0

    # Clean transcript: only non-hallucinated segments
    clean_segments = [r for r in results if not r["is_hallucination"]]
    full_text = " | ".join(r["text"] for r in clean_segments if r["text"])

    return {
        "wav_file"       : wav_path.name,
        "n_vad_segments" : len(segs_to_process),
        "n_transcribed"  : len(clean_segments),
        "n_hallucinated" : len(results) - len(clean_segments),
        "segments"       : results,
        "full_text"      : full_text,
        "total_speech_s" : total_speech_audio,
        "elapsed_s"      : elapsed,
        "rtf"            : rtf,
    }


# ── Test on 3.wav first ───────────────────────────────────────────────────────
print("SEGMENT-LEVEL TRANSCRIPTION TEST — 3.wav")
print("="*60)

wav3_path = AUDIO_DIR / "3.wav"
vad3_row  = df_vad[df_vad["wav_file"] == "3.wav"].iloc[0].to_dict()

print(f"VAD found {vad3_row['n_speech_segs']} segments, "
      f"{vad3_row['total_speech_s']:.1f}s of speech")
print(f"Processing only segments ≥ 1.5s duration...\n")

result3 = transcribe_segments(wav3_path, vad3_row, min_seg_duration=1.5)

print(f"Results:")
print(f"  Segments attempted  : {result3['n_vad_segments']}")
print(f"  Clean transcriptions: {result3['n_transcribed']}")
print(f"  Hallucinations      : {result3['n_hallucinated']}")
print(f"  RTF (speech-only)   : {result3['rtf']:.3f}")
print(f"\nClean transcript:\n  {result3['full_text']}")

print(f"\nAll segments (first 15):")
for seg in result3["segments"][:15]:
    flag = "✅" if not seg["is_hallucination"] else "✗ "
    atc  = "🎙" if seg["has_atc_keyword"] else "  "
    print(f"  {flag}{atc} [{seg['start']:6.1f}s–{seg['end']:6.1f}s] "
          f"{seg['text'][:80]}")

SEGMENT-LEVEL TRANSCRIPTION TEST — 3.wav
VAD found 129 segments, 41.1s of speech
Processing only segments ≥ 0.5s duration...

Results:
  Segments attempted  : 6
  Clean transcriptions: 1
  Hallucinations      : 5
  RTF (speech-only)   : 0.048

Clean transcript:
  Golf Bravo runway two six five point five roger cocks

All segments (first 15):
  ✅🎙 [   0.0s–   5.9s] Golf Bravo runway two six five point five roger cocks
  ✗    [  34.4s–  37.3s] you
  ✗    [  37.4s–  39.9s] you
  ✗    [  40.0s–  42.2s] you
  ✗    [  47.5s–  51.0s] you
  ✗    [  52.8s–  55.9s] you


In [9]:
# ── Callout analysis for 3.wav ────────────────────────────────────────────────
print("\nCALLOUT ANALYSIS — 3.wav")
print("="*55)

transcript = "Golf Bravo runway two six five point five roger cocks"

# Cross-reference with ADS-B data
df_adsb = pd.read_csv(CALLOUTS_CSV, parse_dates=["timestamp"])

# Find ADS-B pings near the audio segment time
# 3.wav starts at ~11:53 local (from manifest) + 0-6 seconds offset
df_manifest = pd.read_csv(MANIFEST_CSV)
wav3_manifest = df_manifest[df_manifest["wav_file"] == "3.wav"]

if len(wav3_manifest) > 0:
    seg_start = pd.Timestamp(wav3_manifest.iloc[0]["start_time"])
    seg_end   = seg_start + pd.Timedelta(seconds=6)

    nearby_adsb = df_adsb[
        (df_adsb["timestamp"] >= seg_start) &
        (df_adsb["timestamp"] <= seg_end + pd.Timedelta(seconds=30))
    ][["Tail","timestamp","Altitude","Speed","range_nm","phase","expected_callout"]]

    print(f"Audio segment: {seg_start.time()} → {seg_end.time()}")
    print(f"\nADS-B pings in that window:")
    if len(nearby_adsb) > 0:
        display(nearby_adsb.head(15))

        # Which aircraft tail contains "GB"?
        gb_aircraft = df_adsb[df_adsb["Tail"].str.contains("GB", na=False)]
        if len(gb_aircraft) > 0:
            print(f"\nAircraft with 'GB' in tail number:")
            display(gb_aircraft[["Tail","timestamp","range_nm",
                                  "phase","expected_callout"]].head(5))
    else:
        print("No ADS-B pings found in window")
        print("(3.wav starts at 11:53 — audio starts before ADS-B trigger zone)")
        print("This is consistent with aircraft just outside 10km trigger range")


CALLOUT ANALYSIS — 3.wav
Audio segment: 11:53:33.138336 → 11:53:39.138336

ADS-B pings in that window:


,Tail,timestamp,Altitude,Speed,range_nm,phase,expected_callout
20218,MLN285,2020-10-22 11:53:54.551000,11550.0,309.0,15.280469,TRANSITING,NaN
20219,MLN285,2020-10-22 11:53:56.087000,11625.0,309.0,15.227675,TRANSITING,NaN
20220,MLN285,2020-10-22 11:53:56.728000,11625.0,309.0,15.227675,TRANSITING,NaN
20221,MLN285,2020-10-22 11:53:58.152000,11800.0,311.0,15.079887,TRANSITING,NaN
20222,MLN285,2020-10-22 11:53:59.128000,11900.0,312.0,15.005302,TRANSITING,NaN
20223,MLN285,2020-10-22 11:54:00.651000,12000.0,312.0,14.979846,TRANSITING,NaN
20224,MLN285,2020-10-22 11:54:01.296999,12025.0,313.0,14.923914,TRANSITING,NaN
20225,MLN285,2020-10-22 11:54:02.633000,12150.0,313.0,14.830027,TRANSITING,NaN
20226,MLN285,2020-10-22 11:54:04.496999,12250.0,313.0,14.741070,TRANSITING,NaN
20227,MLN285,2020-10-22 11:54:05.533999,12300.0,316.0,14.713738,TRANSITING,NaN



Aircraft with 'GB' in tail number:


,Tail,timestamp,range_nm,phase,expected_callout
71235,N318GB,2020-10-22 15:47:00.765000,16.855760,TRANSITING,NaN
71236,N318GB,2020-10-22 15:47:01.804999,16.810036,TRANSITING,NaN
71237,N318GB,2020-10-22 15:47:02.845000,16.810036,TRANSITING,NaN
71238,N318GB,2020-10-22 15:47:03.027000,16.810036,TRANSITING,NaN
71239,N318GB,2020-10-22 15:47:03.027000,16.810036,TRANSITING,NaN
